## Semaphore in Python

Suppose your application has a database connection pool with 3 available connections.

You have 10 threads requesting connections simultaneously.

You don't want all 10 threads accessing the limited resource at once. You want to allow a maximum of 3 threads at a time.

A normal Lock allows only one thread at a time. We need something that allows a fixed number.

A Semaphore is a synchronization mechanism that allows a specified number of threads or couroutines to access a resource simultaneously.

In [1]:
import threading
import time

semaphore = threading.Semaphore(3)

def task(name):
    with semaphore:
        print(name, "entered")
        time.sleep(3)
        print(name, "exited")

threads = []

for i in range(10):
    t = threading.Thread(target=task, args=(f"Thread {i}",))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

Thread 0 entered
Thread 1 entered
Thread 2 entered
Thread 2 exited
Thread 0 exited
Thread 3 entered
Thread 4 entered
Thread 1 exited
Thread 5 entered
Thread 4 exited
Thread 3 exited
Thread 6 entered
Thread 7 entered
Thread 5 exited
Thread 8 entered
Thread 8 exited
Thread 9 entered
Thread 6 exited
Thread 7 exited
Thread 9 exited


In [ ]:
from fastapi import FastAPI
import asyncio
import httpx

app = FastAPI()

# Allow only 3 concurrent calls
ai_semaphore = asyncio.Semaphore(3)

@app.get("/generate")
async def generate():

    async with ai_semaphore:

        print("Acquired semaphore")

        async with httpx.AsyncClient() as client:
            response = await client.get(
                "https://httpbin.org/delay/3"
            )

        return {"result": response.json()}

so when 10 rqsts come ,3 are using llm other waits ,but still event loop is not blocked

## DeadLocks
 A deadlock occurs when two or more threads wait indefinitely for resources held by one another.

In [ ]:
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()

def task1():
    with lock_a:
        print("T1 acquired Lock A")
        time.sleep(1)

        with lock_b:
            print("T1 acquired Lock B")

def task2():
    with lock_b:
        print("T2 acquired Lock B")
        time.sleep(1)

        with lock_a:
            print("T2 acquired Lock A")

t1 = threading.Thread(target=task1)
t2 = threading.Thread(target=task2)

t1.start()
t2.start()

t1.join()
t2.join()

Solution:One common solution is consistent lock ordering.
Always acquire Lock A before Lock B.

In [2]:
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()

def task1():
    with lock_a:
        print("T1 acquired Lock A")
        time.sleep(1)

        with lock_b:
            print("T1 acquired Lock B")
            print("T1 working")

def task2():
    with lock_a:
        print("T2 acquired Lock A")
        time.sleep(1)

        with lock_b:
            print("T2 acquired Lock B")
            print("T2 working")

t1 = threading.Thread(target=task1)
t2 = threading.Thread(target=task2)

t1.start()
t2.start()

t1.join()
t2.join()

print("Both threads completed")

T1 acquired Lock A
T1 acquired Lock B
T1 working
T2 acquired Lock A
T2 acquired Lock B
T2 working
Both threads completed
